# Exploring Demand Split Boundaries and Duplicate Keys

This notebook turns [`scripts/learner_demand_time_audit.py`](../scripts/learner_demand_time_audit.py) into an interactive investigation. Run cells from top to bottom with **Shift + Enter**, then change the examples or filters to explore further.

The central distinction is:

- **Timestamp overlap**: two splits contain rows at the same pickup hour.
- **Duplicate zone-hour key**: two rows have both the same `PULocationID` and the same `pickup_hour`.

A shared hour does not by itself prove that any zone-hour key is repeated. We will inspect each question separately.

## 1. Load the persisted train, validation, and test data

The script locates data relative to its own file. A notebook has no `__file__`, so this setup searches the current directory and its parents for the project root. It works when Jupyter starts in the repository root or in `notebooks/`.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "features").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

FEATURES_DIR = ROOT / "data" / "features"
TIME_COL = "pickup_hour"
ZONE_COL = "PULocationID"
TARGET_COL = "pickup_count"
SPLIT_NAMES = ("train", "val", "test")

if not FEATURES_DIR.is_dir():
    raise FileNotFoundError(
        "Could not find data/features. Start Jupyter inside this project or one of its subfolders."
    )

splits = {
    name: pd.read_parquet(FEATURES_DIR / f"demand_{name}.parquet")
    for name in SPLIT_NAMES
}
print(f"Project root: {ROOT}")
print(f"Loaded splits: {', '.join(splits)}")

Start with the schema and a few rows. In these feature files, one row represents one zone at one pickup hour; `pickup_count` is the demand value for that zone-hour.

In [ ]:
print("Training columns:")
print(splits["train"].columns.tolist())
display(splits["train"].head())

## 2. Compare split sizes and time coverage

`rows` counts records. `hours` counts distinct timestamps across those records, so it is generally smaller when many zones have data in the same hour. `zones` counts distinct zone IDs.

In [ ]:
summary = pd.DataFrame(
    {
        name: {
            "rows": len(frame),
            "distinct pickup hours": frame[TIME_COL].nunique(),
            "distinct zones": frame[ZONE_COL].nunique(),
            "first pickup hour": frame[TIME_COL].min(),
            "last pickup hour": frame[TIME_COL].max(),
            "distinct zone-hour keys": frame[[ZONE_COL, TIME_COL]].drop_duplicates().shape[0],
        }
        for name, frame in splits.items()
    }
).T
display(summary)

### A small example: same hour versus same zone-hour key

Here two splits share an hour, but each has a different zone. There is timestamp overlap, but no duplicate `(zone, hour)` key. In the second example, both the zone and hour repeat, so both rows are marked as duplicates.

In [ ]:
example_hour = pd.Timestamp("2024-01-01 10:00:00")
different_zones = pd.DataFrame(
    {
        "split": ["train", "val"],
        ZONE_COL: [101, 202],
        TIME_COL: [example_hour, example_hour],
    }
)
different_zones["duplicate zone-hour key"] = different_zones.duplicated(
    subset=[ZONE_COL, TIME_COL], keep=False
)
display(different_zones)
print("Same timestamp appears in both splits:", different_zones[TIME_COL].nunique() == 1)
print("Duplicate zone-hour rows:", different_zones["duplicate zone-hour key"].sum())

In [ ]:
same_zone = different_zones.copy()
same_zone[ZONE_COL] = 101
same_zone["duplicate zone-hour key"] = same_zone.duplicated(
    subset=[ZONE_COL, TIME_COL], keep=False
)
display(same_zone)
print("Duplicate zone-hour rows:", same_zone["duplicate zone-hour key"].sum())

## 3. Compare the split boundaries

The first/last timestamps describe each split's range. The `<` comparisons below ask whether the next split starts strictly after the previous one ends. If they are equal, the boundary timestamp is shared; this does not yet tell us which zones occur at that timestamp.

In [ ]:
train_last = splits["train"][TIME_COL].max()
val_first = splits["val"][TIME_COL].min()
val_last = splits["val"][TIME_COL].max()
test_first = splits["test"][TIME_COL].min()

boundary_summary = pd.DataFrame(
    [
        {
            "adjacent splits": "train / val",
            "previous split last hour": train_last,
            "next split first hour": val_first,
            "strictly increasing?": train_last < val_first,
            "same boundary timestamp?": train_last == val_first,
        },
        {
            "adjacent splits": "val / test",
            "previous split last hour": val_last,
            "next split first hour": test_first,
            "strictly increasing?": val_last < test_first,
            "same boundary timestamp?": val_last == test_first,
        },
    ]
).set_index("adjacent splits")
display(boundary_summary)

## 4. Label and combine rows without changing the source frames

Adding a `split` column to copies preserves the originally loaded DataFrames. Concatenating row-wise lets us investigate a key or timestamp across all three splits. `ignore_index=True` creates one new sequential index for the combined table.

In [ ]:
labelled_frames = []

for split_name, frame in splits.items():
    labelled = frame.copy()
    labelled["split"] = split_name
    labelled_frames.append(labelled)

combined = pd.concat(labelled_frames, ignore_index=True)
print(f"Rows in combined data: {len(combined):,}")
print("Rows across the original splits:", sum(len(frame) for frame in splits.values()))
display(combined[["split", ZONE_COL, TIME_COL, TARGET_COL]].head())

## 5. A. Check for duplicate zone-hour keys

`duplicated(subset=[zone, hour], keep=False)` marks **every occurrence** of a repeated key, including its first occurrence. The split label is deliberately not part of the key: if the same zone-hour is present in two splits, both rows should be reported. A shared hour with different zones is not a duplicate key.

In [ ]:
duplicate_mask = combined.duplicated(
    subset=[ZONE_COL, TIME_COL],
    keep=False,
)
duplicate_rows = combined.loc[duplicate_mask]

duplicate_counts = pd.Series(
    {
        "True (rows belonging to repeated keys)": int(duplicate_mask.sum()),
        "False (rows with unique keys)": int((~duplicate_mask).sum()),
    }
)
display(duplicate_counts.to_frame("row count"))
print(f"Rows in combined data: {len(combined):,}")
print("True + False equals total rows:", int(duplicate_mask.sum() + (~duplicate_mask).sum()) == len(combined))

In [ ]:
# Display all matched rows. An empty table means no duplicate zone-hour key was found.
display(duplicate_rows)

If duplicates exist, inspect which keys repeat and whether occurrences span multiple splits. Grouping by the key gives one row per repeated key; `nunique()` then counts how many different split labels contain that key.

In [ ]:
if duplicate_rows.empty:
    print("No repeated zone-hour keys to group.")
else:
    duplicate_key_summary = (
        duplicate_rows.groupby([ZONE_COL, TIME_COL])
        .agg(
            occurrences=("split", "size"),
            split_count=("split", "nunique"),
            splits=("split", lambda values: ", ".join(sorted(values.unique()))),
        )
        .sort_values(["split_count", "occurrences"], ascending=False)
    )
    display(duplicate_key_summary)

## 6. B. Inspect the shared boundary hours

For each adjacent pair, select every row at the shared timestamp from the combined data. The table shows its split, zone, timestamp, and demand count. The row count per split and distinct-zone count help tell whether the same hour is represented by different zones or overlapping zones.

In [ ]:
boundary_hours = [
    ("train/val", train_last),
    ("val/test", val_last),
]
boundary_columns = ["split", ZONE_COL, TIME_COL, TARGET_COL]
boundary_frames = {}

for boundary_name, boundary_hour in boundary_hours:
    # eq() compares each row's timestamp with the boundary, creating a Boolean mask.
    mask = combined[TIME_COL].eq(boundary_hour)
    # .loc[rows, columns] selects only rows at that hour and the requested fields.
    rows = combined.loc[mask, boundary_columns].sort_values(["split", ZONE_COL])
    boundary_frames[boundary_name] = rows

    print(f"{boundary_name} boundary hour: {boundary_hour}")
    display(rows)

In [ ]:
boundary_count_rows = []

for boundary_name, rows in boundary_frames.items():
    # size() counts rows in each split; reindex() includes splits with no rows as zero.
    per_split = rows.groupby("split").size().reindex(SPLIT_NAMES, fill_value=0)
    boundary_count_rows.append(
        {
            "boundary": boundary_name,
            **{f"{name} rows": int(per_split[name]) for name in SPLIT_NAMES},
            "total rows": len(rows),
            "distinct zones across selected rows": rows[ZONE_COL].nunique(),
        }
    )

boundary_counts = pd.DataFrame(boundary_count_rows).set_index("boundary")
display(boundary_counts)

### Which zones occur on both sides of the boundary?

The following set intersection directly checks zone overlap between each adjacent split at the shared hour. An empty intersection means that the timestamp is shared but there are no duplicate zone-hour keys at that boundary. `len()` counts the intersection; sorting makes the result easy to read.

In [ ]:
adjacent_splits = {
    "train/val": ("train", "val"),
    "val/test": ("val", "test"),
}

for boundary_name, rows in boundary_frames.items():
    left_name, right_name = adjacent_splits[boundary_name]
    left_zones = set(rows.loc[rows["split"].eq(left_name), ZONE_COL])
    right_zones = set(rows.loc[rows["split"].eq(right_name), ZONE_COL])
    shared_zones = left_zones & right_zones

    print(f"{boundary_name}: {len(shared_zones)} zone(s) occur in both splits")
    print("Shared zone IDs:", sorted(shared_zones))

## 7. Explore a single boundary interactively

Change `boundary_to_explore` to `"train/val"` or `"val/test"`. Then filter to one zone ID to compare the exact split membership and demand at that hour. If the zone is absent from one side, the filtered table will contain only the side where it is present.

In [ ]:
boundary_to_explore = "train/val"  # Try "val/test" too.
example_zone = int(boundary_frames[boundary_to_explore][ZONE_COL].iloc[0])

zone_at_boundary = boundary_frames[boundary_to_explore].loc[
    boundary_frames[boundary_to_explore][ZONE_COL].eq(example_zone)
]
print(f"Rows for zone {example_zone} at the {boundary_to_explore} boundary:")
display(zone_at_boundary)

## 8. What the investigation establishes

- Equal neighboring endpoint timestamps establish **timestamp overlap**.
- A `True` in the duplicate mask establishes that the same **zone and hour** occur more than once in the combined splits.
- The boundary tables show which split contributes each zone-hour row; their per-split counts can differ even when the timestamp is the same.
- The shared-zone intersection is a boundary-specific check. The global duplicate mask checks every timestamp, not only the two boundary hours.

Use the actual tables and counts above to state what is true of the files currently loaded. As an exercise, change the duplicate key to only `[TIME_COL]` and compare the result: that answers a different question, namely which timestamps occur in multiple rows.

In [ ]:
# Optional exercise: count repeated timestamps, ignoring zone identity.
timestamp_mask = combined.duplicated(subset=[TIME_COL], keep=False)
print("Rows whose timestamp appears more than once:", int(timestamp_mask.sum()))
print("Rows whose timestamp is unique:", int((~timestamp_mask).sum()))
display(combined.loc[timestamp_mask, ["split", ZONE_COL, TIME_COL]].head(10))

## 9. Rebuild the splits so each whole hour stays together

The saved splits were cut by row position, so an hour with records for many zones could land on both sides of a cut. Here we choose the 80% and 90% cut positions from the **sorted list of distinct hours**, then use timestamp masks. The start of validation and test is included in that split; the previous split ends strictly before it.

This creates new in-memory frames for learning and verification. It does not write over the saved Parquet files.

In [ ]:
# Remove only the provenance label: source is the row data from all saved splits.
source = combined.drop(columns="split").copy()

# One sorted timestamp per hour, regardless of how many zones have rows in that hour.
hours = (
    source[TIME_COL]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

# Convert fractions into zero-based positions in the distinct-hour list.
train_end = int(len(hours) * 0.8)
val_end = int(len(hours) * 0.9)

# These positions mark the first hour in validation and test, respectively.
validation_start = hours.iloc[train_end]
test_start = hours.iloc[val_end]

print(f"Rows to partition: {len(source):,}")
print(f"Distinct hours: {len(hours):,}")
print(f"80% cut position {train_end}: validation starts at {validation_start}")
print(f"90% cut position {val_end}: test starts at {test_start}")

### Make one Boolean mask per split

Each comparison creates a True/False value for every row. Parentheses around each comparison are important: pandas combines elementwise conditions with `&`; Python's scalar `and` is not suitable for Series. The validation interval includes `validation_start` but excludes `test_start`, so no hour is assigned to two splits.

In [ ]:
# Earlier timestamps belong to training.
training_mask = (source[TIME_COL] < validation_start)

# Validation includes its first hour and stops before the first test hour.
validation_mask = (
    (source[TIME_COL] >= validation_start)
    & (source[TIME_COL] < test_start)
)

# The test interval includes test_start and all later timestamps.
test_mask = (source[TIME_COL] >= test_start)

# .loc selects rows where each mask is True; copy() gives each split its own frame.
strict_splits = {
    "train": source.loc[training_mask].copy(),
    "val": source.loc[validation_mask].copy(),
    "test": source.loc[test_mask].copy(),
}
print("Created strict_splits with keys:", list(strict_splits))

## 10. Compare summaries and check strict time separation

Print the same summary fields as the saved splits. The number of rows can shift slightly between partitions because an entire boundary hour moves together, while the overall records are preserved.

In [ ]:
strict_summary = pd.DataFrame(
    {
        name: {
            "rows": len(frame),
            "hours": frame[TIME_COL].nunique(),
            "zones": frame[ZONE_COL].nunique(),
            "first": frame[TIME_COL].min(),
            "last": frame[TIME_COL].max(),
        }
        for name, frame in strict_splits.items()
    }
).T
display(strict_summary)

train_before_val = (
    strict_splits["train"][TIME_COL].max()
    < strict_splits["val"][TIME_COL].min()
)
val_before_test = (
    strict_splits["val"][TIME_COL].max()
    < strict_splits["test"][TIME_COL].min()
)
print("Train ends strictly before validation starts:", train_before_val)
print("Validation ends strictly before test starts:", val_before_test)

## 11. Verify rows, keys, and timestamp assignment

These are separate checks: row counts detect dropped/added records; duplicate detection checks `(zone, hour)` keys; key-set comparison checks that the original keys are still represented. Finally, grouping by hour and counting distinct split labels directly tests whether any timestamp is split across train, validation, and test. Every count should equal one.

In [ ]:
# Add provenance labels to copies, then stack the new partitions for cross-split checks.
strict_labelled_frames = []
for split_name, frame in strict_splits.items():
    labelled = frame.copy()
    labelled["split"] = split_name
    strict_labelled_frames.append(labelled)

strict_combined = pd.concat(strict_labelled_frames, ignore_index=True)
expected_row_count = 145_728
print(f"New total rows: {len(strict_combined):,}")
print(f"Matches expected 145,728 rows: {len(strict_combined) == expected_row_count}")
print(f"Matches source row count: {len(strict_combined) == len(source)}")

In [ ]:
# keep=False marks every occurrence if a zone-hour key is repeated.
strict_duplicate_mask = strict_combined.duplicated(
    subset=[ZONE_COL, TIME_COL],
    keep=False,
)
print("Duplicate zone-hour rows:", int(strict_duplicate_mask.sum()))
print("No duplicate zone-hour keys:", not strict_duplicate_mask.any())

# A MultiIndex treats zone and hour together as one composite key.
source_keys = pd.MultiIndex.from_frame(source[[ZONE_COL, TIME_COL]])
strict_keys = pd.MultiIndex.from_frame(strict_combined[[ZONE_COL, TIME_COL]])
missing_keys = source_keys.difference(strict_keys)
extra_keys = strict_keys.difference(source_keys)
print("Original keys missing:", len(missing_keys))
print("Unexpected keys added:", len(extra_keys))
print("Every original zone-hour key remains:", len(missing_keys) == 0 and len(extra_keys) == 0)

### The direct timestamp-to-split check

`groupby(TIME_COL)["split"].nunique()` groups all rows for each hour, then counts how many different split labels occur in that group. A result greater than one means that hour appears in multiple splits?the original boundary problem?even if every individual zone-hour key is unique. A result of one for every hour confirms each timestamp is assigned to exactly one split.

In [ ]:
timestamp_split_counts = strict_combined.groupby(TIME_COL)["split"].nunique()
display(timestamp_split_counts.value_counts().rename_axis("number of splits for an hour").to_frame("number of hours"))
every_hour_in_one_split = timestamp_split_counts.eq(1).all()
print("Every timestamp belongs to exactly one split:", every_hour_in_one_split)

# Compare with the saved splits: these counts expose any hours divided across splits.
original_timestamp_split_counts = combined.groupby(TIME_COL)["split"].nunique()
print("Saved splits: number of hours found in multiple splits:", int(original_timestamp_split_counts.gt(1).sum()))
print("Strict splits: number of hours found in multiple splits:", int(timestamp_split_counts.gt(1).sum()))